In [2]:
import pandas as pd
import requests
import numpy
from dotenv import load_dotenv
import os

Create a list that includes the top 200 rated movies on themoviedb.org using an API key

In [11]:
load_dotenv()

api = os.environ.get("api_key")
url = "https://api.themoviedb.org/3/movie/top_rated"

top_movies = []
for page in range(1, 14):
    params = {
        "api_key": api,
        "language": "en-US",
        "page": page
    }
    
    response = requests.get(url, params=params)
    movies = response.json()
    
    if "results" in movies:
        top_movies.extend(movies["results"])

top_200 = top_movies[:200]

    

Get genre mapping from the website (ex. 16 for its genre ID is Animation) and create genre dictionary to map the genre ID to an actual genre.

In [22]:
genre_url = "https://api.themoviedb.org/3/genre/movie/list"
params = {
    "api_key": api,
    "language": "en-US"
}

response_genre = requests.get(genre_url, params=params)
genre_data = response_genre.json()
genre_dict = {str(genre["id"]): genre["name"] for genre in genre_data.get("genres", [])}

Creating the raw dataset and saving it under the "raw" folder

In [24]:
top_200_movies = pd.DataFrame(top_200)
top_200_movies.to_csv("raw/top_200_raw.csv", index=False)

Add a genre column and create a new, clean dataset that includes five variables that we want to focus on.

In [25]:
top_200_movies["genre_ids"] = top_200_movies["genre_ids"].str[0].astype(str)
top_200_movies["genre"] = top_200_movies["genre_ids"].map(genre_dict)
top_200_movies_clean = top_200_movies[["title", "release_date", "overview", "genre", "vote_average"]].copy()
top_200_movies_clean["release_date"] = pd.to_datetime(top_200_movies_clean["release_date"])
top_200_movies_clean.to_csv("processed/top_200_clean.csv", index=False)